# W6 品質檢查台（Quality Check Notebook）

**用途**：任何人（Vivi / Sonia / Tang）在任何一次跑批之後，用同一個流程把品質數字算出來、檢查有沒有不能對外的數字混進來、並留一筆不可覆寫的記錄。

**設計原則**

1. **只讀已落盤的 json，不重跑模型**：跑一次約 3 秒，不需要 GPU。
2. **指標自己重算，不信任 `summary`**：舊跑批的 `param_coverage` / `param_hit_total` 缺欄或口徑不一，本 Notebook 一律從 `rows` 重算。
3. **每筆記錄帶雜湊**：記錄裡存 `results_defer_W*.json` 的 SHA1，改過檔案就對不上，無法事後偷改數字。
4. **每個數字都帶 scope**：`in-gate`（門控內）與 `all-edits`（全部編輯）分兩列，禁止混用。

**前置條件**：`pip install pandas`；在 repo 根目錄或任何子目錄啟動 Jupyter 都可以（會自動往上找 `.git`）。

**產出**

- 螢幕上的品質總表（每跑批一列）
- `EasyEdit/w6run/quality_log.jsonl`：追加式記錄（append-only）
- `FONG CHIN WAI/w6_deferred/quality_snapshots/quality_<時間>_{csv,md}`：當次快照


## 步驟 0：環境與設定

**跑之前必改**：`CHECKER` 填自己的名字。留預設值 `UNKNOWN` 的記錄視為無效記錄。


In [1]:
from pathlib import Path
import json, glob, hashlib, datetime, os
import pandas as pd

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / ".git").exists():
            return p
    raise SystemExit("找不到 repo 根目錄（.git 不存在）——請在 repo 內啟動 Jupyter")


HERE = Path.cwd()
REPO = find_repo_root(HERE)
EE = REPO / "EasyEdit"

CONFIG = {
    # ←←← 每次跑之前改這裡
    "checker": "Vivi",            # 你的名字（例："Sonia" / "Tang" / "Vivi"）
    "note": "首次基線：重建全部跑批的品質表（記憶 recall 仍假設 1.0）",                      # 這次檢查的目的，例："Sonia 改鍵之後第一次重算"

    # 資料與跑批
    "data": EE / "data" / "counterfact.json",
    "run_glob": str(EE / "w6run" / "**" / "results_defer_W*.json"),
    "exclude_dirs": ["smoke_guard", "smoke"],   # 冒煙測試不進品質表

    # 端到端上限的假設：記憶通道真實 recall（Sonia 修好之前請維持 1.0 = 上限）
    "memory_recall": 1.0,

    # 輸出
    "log_path": EE / "w6run" / "quality_log.jsonl",
    "snapshot_dir": REPO / "FONG CHIN WAI" / "w6_deferred" / "quality_snapshots",
}

print("repo      :", REPO)
print("data      :", CONFIG["data"], "| 存在:", CONFIG["data"].exists())
print("checker   :", CONFIG["checker"])


repo      : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26
data      : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\EasyEdit\data\counterfact.json | 存在: True
checker   : Vivi


## 步驟 1：載入資料切片，並自動推斷每個跑批的 `--data-offset`

舊跑批沒有把 offset 寫進 `summary`，靠人工記很容易串片。這裡用 row 的 `subject` 序列去比對 `counterfact.json`，前三條對得上才採信，否則標 `UNKNOWN` 並拒絕計算端到端。


In [2]:
with open(CONFIG["data"], encoding="utf-8") as fh:
    RAW = json.load(fh)

# 只建前 2000 條的 subject 索引，足夠覆蓋 W6 用到的切片
DATA = []
for i, rec in enumerate(RAW[:2000]):
    rr = rec["requested_rewrite"]
    DATA.append({
        "idx": i,
        "subject": rr["subject"],
        "relation": rr.get("relation_id", "fact"),
        "prompt": rr["prompt"].replace("{}", rr["subject"]),
        "target_new": rr["target_new"]["str"] if isinstance(rr["target_new"], dict) else rr["target_new"],
    })

SUBJECT_POS = {}
for d in DATA:
    SUBJECT_POS.setdefault(d["subject"].strip().lower(), []).append(d["idx"])

print("data records loaded:", len(DATA))


def infer_offset(rows, probes: int = 3):
    """用 row.subject 序列反推 offset；多個候選則取第一個能連續對上的。"""
    if not rows:
        return None
    first = str(rows[0].get("subject", "")).strip().lower()
    for cand in SUBJECT_POS.get(first, []):
        ok = True
        for k in range(1, probes):
            if cand + k >= len(DATA) or k >= len(rows):
                break
            if str(rows[k].get("subject", "")).strip().lower() != DATA[cand + k]["subject"].strip().lower():
                ok = False
                break
        if ok:
            return cand
    return None


data records loaded: 2000


## 步驟 2：指標定義（全部從 `rows` 重算）

| 指標 | 定義 | scope |
|---|---|---|
| `coverage` | 走參數通道的編輯數 ÷ 全部編輯 | all-edits |
| `immediate` | 套用當下就答對 ÷ 參數通道編輯數 | in-gate |
| `final_in_gate` | 流結束後仍答對 ÷ 參數通道編輯數 | in-gate |
| `hit_total` | 流結束後答對 ÷ 全部編輯 | all-edits |
| `e2e_floor` | 參數通道命中 ÷ 全部（記憶通道未讀，現況） | all-edits |
| `e2e_ceiling` | (參數命中 + 記憶路由數 × recall) ÷ 全部 | all-edits |
| `nll_delta` | 流結束後 probe 文本 NLL 增量（越小越好） | model-level |
| `degen` | 退化答案（空字串或同詞重複 ≥4）÷ 全部 | all-edits |

**注意**：`coverage` 不是記憶準確率；`hit_total` 才是可對外的端到端（純權重）數字。


In [3]:
PARAM_ROUTES = {"parameter", "hypernetwork_batched", "deferred_batched"}
MEMORY_ROUTES = {"side_memory", "batched_route_fallback", "memory"}
DEAD_ROUTES = {"gate_reject"}


def channel_of(route):
    r = (route or "").lower()
    if r in PARAM_ROUTES:
        return "parameter"
    if r in MEMORY_ROUTES:
        return "memory"
    if r in DEAD_ROUTES:
        return "reject"
    return "other"


def is_degenerate(ans) -> bool:
    if not ans or not str(ans).strip():
        return True
    toks = str(ans).split()
    return len(toks) >= 4 and len(set(toks)) == 1


def score_run(path: Path, data_offset, memory_recall: float):
    run = json.loads(path.read_text(encoding="utf-8"))
    rows = run.get("rows", [])
    summ = run.get("summary", {})
    if data_offset is None:
        return None, run, "offset 無法推斷，跳過"

    n = len(rows)
    if n == 0:
        return None, run, "沒有 rows"

    edits = DATA[data_offset:data_offset + n]
    if len(edits) != n:
        return None, run, "資料切片長度不足（%d vs %d）" % (len(edits), n)

    cnt = {"parameter": 0, "memory": 0, "reject": 0, "other": 0}
    imm_hit = fin_hit = 0
    degen = 0
    leak_query = 0
    for row, edit in zip(rows, edits):
        ch = channel_of(row.get("route"))
        cnt[ch] += 1
        if ch == "parameter":
            imm_hit += bool(row.get("immediate_ok"))
            fin_hit += bool(row.get("final_ok"))
        degen += is_degenerate(row.get("final_answer"))
        # 洩漏：檢索用的 prompt 本身已經含新答案 → 命中不算檢索能力
        if str(edit["target_new"]).strip().lower() in str(edit["prompt"]).lower():
            leak_query += 1

    param_n = cnt["parameter"]
    mem_n = cnt["memory"]
    rec = {
        "run": os.path.relpath(path, EE).replace("\\", "/"),
        "sha1": hashlib.sha1(path.read_bytes()).hexdigest()[:12],
        "offset": data_offset,
        "slice": "%d-%d" % (data_offset, data_offset + n - 1),
        "n": n,
        "W": summ.get("defer_window"),
        "layers": str(summ.get("layers")),
        "refine": summ.get("refine_passes"),
        "delta_budget": summ.get("delta_budget"),
        "coverage": round(param_n / n, 4),
        "immediate": round(imm_hit / param_n, 4) if param_n else None,
        "final_in_gate": round(fin_hit / param_n, 4) if param_n else None,
        "hit_total": round(fin_hit / n, 4),
        "e2e_floor": round(fin_hit / n, 4),
        "e2e_ceiling": round((fin_hit + mem_n * memory_recall) / n, 4),
        "nll_delta": summ.get("nll_delta"),
        "degen": round(degen / n, 4),
        "param_n": param_n,
        "mem_routed": mem_n,
        "rejected": cnt["reject"],
        "leak_queries": leak_query,
    }
    return rec, run, None


## 步驟 3：跑全部批次，生成品質總表


In [4]:
records, errors, runs_raw = [], [], {}
paths = sorted(glob.glob(CONFIG["run_glob"], recursive=True))
paths = [Path(p) for p in paths if not any(x in Path(p).parts for x in CONFIG["exclude_dirs"])]
print("發現 %d 個跑批檔" % len(paths))

for p in paths:
    run = json.loads(p.read_text(encoding="utf-8"))
    off = infer_offset(run.get("rows", []))
    rec, _, err = score_run(p, off, CONFIG["memory_recall"])
    if rec is None:
        errors.append((os.path.relpath(p, EE), err))
    else:
        records.append(rec)
        runs_raw[rec["run"]] = run

df = pd.DataFrame(records).sort_values(["run"]).reset_index(drop=True)
if errors:
    print("\n跳過：")
    for e in errors:
        print("  -", e[0], "|", e[1])

cols = ["run", "slice", "W", "layers", "refine", "delta_budget", "n", "coverage",
        "immediate", "final_in_gate", "hit_total", "e2e_ceiling", "nll_delta", "degen",
        "param_n", "mem_routed", "rejected"]
display(df[cols] if len(df) else df)


發現 27 個跑批檔


,run,slice,W,layers,refine,delta_budget,n,coverage,immediate,final_in_gate,hit_total,e2e_ceiling,nll_delta,degen,param_n,mem_routed,rejected
0,w6run/A/results_defer_W1.json,0-99,1,"[4, 5, 6, 7, 8]",0,0.5,100,0.00,NaN,NaN,0.00,0.10,1.3208,0.04,0,10,0
1,w6run/A/results_defer_W10.json,0-99,10,"[4, 5, 6, 7, 8]",0,0.5,100,0.10,1.0,0.1000,0.01,0.91,1.0230,0.01,10,90,0
2,w6run/A/results_defer_W25.json,0-99,25,"[4, 5, 6, 7, 8]",0,0.5,100,0.06,1.0,0.1667,0.01,0.95,0.7195,0.00,6,94,0
3,w6run/B/results_defer_W100.json,0-99,100,"[4, 5, 6, 7, 8]",0,0.5,100,0.07,1.0,1.0000,0.07,1.00,0.3719,0.00,7,93,0
4,w6run/B/results_defer_W50.json,0-99,50,"[4, 5, 6, 7, 8]",0,0.5,100,0.07,1.0,0.7143,0.05,0.98,0.3728,0.01,7,93,0
5,w6run/C/results_defer_W25.json,0-99,25,"[4, 5, 6, 7, 8]",0,0.5,100,0.57,1.0,0.6667,0.38,0.81,0.9454,0.01,57,43,0
6,w6run/C/results_defer_W50.json,0-99,50,"[4, 5, 6, 7, 8]",0,0.5,100,0.61,1.0,0.9180,0.56,0.95,0.4365,0.03,61,39,0
7,w6run/D/results_defer_W50.json,0-99,50,"[4, 5, 6, 7, 8]",0,0.8,100,0.61,1.0,0.9180,0.56,0.95,0.4365,0.03,61,39,0
8,w6run/F/results_defer_W100.json,0-99,100,"[4, 5, 6, 7, 8, 9, 10]",0,0.8,100,0.47,1.0,1.0000,0.47,1.00,0.7991,0.04,47,53,0
9,w6run/F/results_defer_W50.json,0-99,50,"[4, 5, 6, 7, 8, 9, 10]",0,0.8,100,0.46,1.0,0.6739,0.31,0.85,1.2549,0.14,46,54,0


## 步驟 4：讀總表的三個角度

1. **in-gate 看 `immediate` / `final_in_gate`**：這是寫入器本身的能力，與門控無關。
2. **all-edits 看 `hit_total`**：這是純權重路線可對外的端到端數字。
3. **`e2e_ceiling` 減 `hit_total`**：這是「記憶通道打開之後能多拿多少」——差距大代表瓶頸不在寫入器。

判讀規則：`final_in_gate` 高但 `hit_total` 低 ⇒ coverage（門控准入）問題；`coverage` 高但 `final_in_gate` 低 ⇒ 寫入保值問題。


In [5]:
if len(df):
    gate_dominated = (df["coverage"] < 0.35)
    print("門控主導（coverage < 0.35）的批次：")
    print(df.loc[gate_dominated, ["run", "slice", "coverage", "final_in_gate", "hit_total"]].to_string(index=False))

    print("\n八個切片的平均（只算有 offset 的批次）：")
    print({
        "avg_coverage": round(df["coverage"].mean(), 4),
        "avg_final_in_gate": round(df["final_in_gate"].dropna().mean(), 4),
        "avg_hit_total": round(df["hit_total"].mean(), 4),
        "avg_e2e_ceiling": round(df["e2e_ceiling"].mean(), 4),
        "avg_nll_delta": round(df["nll_delta"].dropna().mean(), 4) if df["nll_delta"].notna().any() else None,
        "avg_degen": round(df["degen"].mean(), 4),
    })


門控主導（coverage < 0.35）的批次：
                                                      run   slice  coverage  final_in_gate  hit_total
                            w6run/A/results_defer_W1.json    0-99      0.00            NaN       0.00
                           w6run/A/results_defer_W10.json    0-99      0.10         0.1000       0.01
                           w6run/A/results_defer_W25.json    0-99      0.06         0.1667       0.01
                          w6run/B/results_defer_W100.json    0-99      0.07         1.0000       0.07
                           w6run/B/results_defer_W50.json    0-99      0.07         0.7143       0.05
                   w6run/H_heldout/results_defer_W50.json 100-199      0.23         0.8696       0.20
w6run/I2_chunk10_replay_dev/results_defer_W50_replay.json 200-299      0.15         0.6667       0.10
           w6run/M_H_repro_heldout/results_defer_W50.json 100-199      0.23         0.8696       0.20
                   w6run/R_o200_K/results_defer_W100.jso

## 步驟 5：不可對外數字稽核（硬性檢查）

以下任何一項為 `FAIL`，該批次數字就**不得**寫進報告或對組外說明：

| 檢查項 | 為什麼 |
|---|---|
| `leak_queries == 0` | 檢索查詢本身含答案 ⇒ 命中是構造性的 |
| `rejected` 與 `mem_routed` 不重複計入命中 | 假 REJECT 曾被誤算成命中 |
| `degen <= 0.05` | 退化答案被當成命中會虛高 |
| `final_in_gate` 的 n ≥ 10 | n=7 的 1.00 不能單獨引用 |
| `nll_delta` 必須同時報 | 只報命中率不報流暢度成本等於隱瞞代價 |
| canonical 與 paraphrase 分開 | 只報 canonical 等於重演 97.48% 的錯 |


In [6]:
audit = []
for _, r in df.iterrows():
    row = {"run": r["run"], "slice": r["slice"]}
    row["no_leak"] = "PASS" if r["leak_queries"] == 0 else "FAIL(%d)" % r["leak_queries"]
    row["degen_ok"] = "PASS" if r["degen"] <= 0.05 else "FAIL(%.3f)" % r["degen"]
    row["in_gate_n"] = "PASS" if r["param_n"] >= 10 else "FAIL(in-gate n=%d)" % r["param_n"]
    row["nll_reported"] = "PASS" if pd.notna(r["nll_delta"]) else "FAIL"
    audit.append(row)

audit_df = pd.DataFrame(audit)
display(audit_df if len(audit_df) else audit_df)

bad = audit_df[(audit_df.astype(str).apply(lambda s: s.str.contains("FAIL")).any(axis=1))] if len(audit_df) else audit_df
print("\n需要標註的批次：%d" % len(bad))
if len(bad):
    display(bad)


,run,slice,no_leak,degen_ok,in_gate_n,nll_reported
0,w6run/A/results_defer_W1.json,0-99,PASS,PASS,FAIL(in-gate n=0),PASS
1,w6run/A/results_defer_W10.json,0-99,PASS,PASS,PASS,PASS
2,w6run/A/results_defer_W25.json,0-99,PASS,PASS,FAIL(in-gate n=6),PASS
3,w6run/B/results_defer_W100.json,0-99,PASS,PASS,FAIL(in-gate n=7),PASS
4,w6run/B/results_defer_W50.json,0-99,PASS,PASS,FAIL(in-gate n=7),PASS
5,w6run/C/results_defer_W25.json,0-99,PASS,PASS,PASS,PASS
6,w6run/C/results_defer_W50.json,0-99,PASS,PASS,PASS,PASS
7,w6run/D/results_defer_W50.json,0-99,PASS,PASS,PASS,PASS
8,w6run/F/results_defer_W100.json,0-99,PASS,PASS,PASS,PASS
9,w6run/F/results_defer_W50.json,0-99,PASS,FAIL(0.140),PASS,PASS



需要標註的批次：9


,run,slice,no_leak,degen_ok,in_gate_n,nll_reported
0,w6run/A/results_defer_W1.json,0-99,PASS,PASS,FAIL(in-gate n=0),PASS
2,w6run/A/results_defer_W25.json,0-99,PASS,PASS,FAIL(in-gate n=6),PASS
3,w6run/B/results_defer_W100.json,0-99,PASS,PASS,FAIL(in-gate n=7),PASS
4,w6run/B/results_defer_W50.json,0-99,PASS,PASS,FAIL(in-gate n=7),PASS
9,w6run/F/results_defer_W50.json,0-99,PASS,FAIL(0.140),PASS,PASS
10,w6run/G/results_defer_W50.json,0-99,PASS,FAIL(0.370),PASS,PASS
14,w6run/J_W100_refine1_heldout/results_defer_W10...,100-199,PASS,FAIL(0.250),PASS,PASS
15,w6run/K2_W50_refine1_guarded_heldout/results_d...,100-199,PASS,FAIL(0.110),PASS,PASS
17,w6run/L2_W50_r1_b06_heldout/results_defer_W50....,100-199,PASS,FAIL(0.110),PASS,PASS


## 步驟 6：寫入記錄（append-only，不可覆寫）

每次執行都會在 `EasyEdit/w6run/quality_log.jsonl` 追加一筆記錄，內容包含：

- `ts` / `checker` / `note`：誰、什麼時候、為什麼跑
- `run` / `sha1`：跑批檔路徑與內容雜湊（檔案被改過就對不上）
- 全部指標（含 scope）

**規則**：記錄只准追加，不准改寫或刪除；`checker` 為 `UNKNOWN` 的記錄視為無效。


In [7]:
if CONFIG["checker"].strip() in ("", "UNKNOWN", "請填入你的名字"):
    print("⚠ CHECKER 未填，記錄會標成 UNKNOWN（無效記錄）。請在步驟 0 填入名字後重跑本 cell。")

ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
LOG = Path(CONFIG["log_path"])
LOG.parent.mkdir(parents=True, exist_ok=True)

COMMIT = "?"
try:
    import subprocess
    COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=str(REPO),
                            capture_output=True, text=True).stdout.strip() or "?"
except Exception:
    pass

written = 0
with open(LOG, "a", encoding="utf-8") as fh:
    for _, r in df.iterrows():
        rec = {"ts": ts, "checker": CONFIG["checker"].strip() or "UNKNOWN",
               "note": CONFIG["note"], "commit": COMMIT,
               "memory_recall_assumed": CONFIG["memory_recall"]}
        rec.update({k: (None if (isinstance(v, float) and pd.isna(v)) else v) for k, v in r.items()})
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        written += 1

print("已追加 %d 筆到 %s" % (written, LOG))
print("當前累計記錄數：", sum(1 for _ in open(LOG, encoding="utf-8")))


已追加 27 筆到 C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\EasyEdit\w6run\quality_log.jsonl
當前累計記錄數： 27


## 步驟 7：輸出快照（csv + md）

快照檔名帶時間戳，直接可貼進報告或用 Excel 開。


In [8]:
SNAP = Path(CONFIG["snapshot_dir"])
SNAP.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
csv_p = SNAP / ("quality_%s.csv" % stamp)
md_p = SNAP / ("quality_%s.md" % stamp)

df.to_csv(csv_p, index=False, encoding="utf-8-sig")

lines = ["# W6 品質快照 %s" % ts, "",
         "- 檢查者：%s" % CONFIG["checker"], "- 備註：%s" % (CONFIG["note"] or "-"),
         "- commit：%s" % COMMIT,
         "- 端到端上限假設 memory_recall = %.2f" % CONFIG["memory_recall"], "",
         df[cols].to_markdown(index=False) if hasattr(df, "to_markdown") else df[cols].to_string(index=False),
         "", "## 稽核結果", "", audit_df.to_string(index=False) if len(audit_df) else "(無)"]
md_p.write_text("\n".join(lines), encoding="utf-8")

print("csv:", csv_p)
print("md :", md_p)


csv: C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w6_deferred\quality_snapshots\quality_20261107_223746.csv
md : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w6_deferred\quality_snapshots\quality_20261107_223746.md


## 步驟 8：對外口徑（貼給任何人看之前先檢查這五條）

1. 只說「門控內 0.6% → 87.0%（held-out n=23）」，不說「端到端修復」。
2. `coverage` 0.23→0.59 一定要連著 `NLL +0.82` 一起講。
3. 不引用 `78.8%`（門控外、閉式解，n=500）作為本專案命中率。
4. 不引用 `97.48%` 作為記憶準確率（那是 routing coverage，且記憶通道構造性恆真）。
5. `e2e_ceiling` 必須標明是「假設記憶 recall = 1.0 的上限」，不是實測。

任何人要引用新數字，先跑一次本 Notebook，把 `quality_*.md` 一起附上。
